In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "volter2018intuitive")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Voelter_2018_exp1_AnimCognit_INWS.csv")
complete_path_2 = os.path.join(original_data_pathway, "Voelter_2018_exp2_AnimCognit_INWS.csv")
complete_path_3 = os.path.join(original_data_pathway, "Voelter_2018_exp3_AnimCognit_INWS.csv")
complete_path_4 = os.path.join(original_data_pathway, "Voelter_2018_exp4_AnimCognit_INWS.csv")
complete_path_5 = os.path.join(original_data_pathway, "Voelter_2018_exp5_AnimCognit_INWS.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
df1 = pd.read_csv(complete_path_1)
df1 = df1.assign(experiment='1')
df2 = pd.read_csv(complete_path_2)
df2 = df2.assign(experiment='2')
df3 = pd.read_csv(complete_path_3)
df3 = df3.assign(experiment='3')
df4 = pd.read_csv(complete_path_4)
df4 = df4.assign(experiment='4')
df5 = pd.read_csv(complete_path_5)
df5 = df5.assign(experiment='5')

In [3]:
data_frames=[df1, df2, df3, df4, df5]

In [4]:
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x=x.rename(columns={"subject": "ape"})
    x=x.rename(columns={"species": "species_original"})
    x=x.rename(columns={"sex": "sex_original"})
    x['study_id']="volter2018intuitive"
    data_frames[index]=x
new_df=data_frames[0]

In [5]:
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [6]:

fulldf['ape'] = fulldf['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')


In [7]:
fulldf = fulldf.rename(columns={"correct_side.1": "correct_side_1",
    "experience with mirror": "experience_with_mirror",
    "pre-experience": "pre_experience"})

fulldf.rename(columns={"ape": "participant", "age":"age_in_years"}, inplace=True)

In [8]:
# float_col = fulldf.select_dtypes(include=['float64'])
# #print(float_col)
# for col in float_col:
#     fulldf[col] = fulldf[col].apply(lambda x: int(x) if x == x else "")
# fulldf.replace('___', np.nan, inplace=True) # if above "" was "___" this makes them floats again, because nan is a float.

In [9]:
fulldf=fulldf[['study_id', 'experiment', 'participant', 'age_in_years', 'sex','species', 
         'phase', 'session', 'trial',
        'condition', 'order', 'order2', 'correct_side', 'correct_side_1','first_response',
       'chosen_side', 'distance', 'visibility', 'baited_side',
       'touch_occluder_at_correct_side', 'order_block', 'order_block2',
       'session_within_block', 'block', 
       'side_with_food_image', 'experience_with_mirror', 'pre_experience']]


In [10]:
for index in range(1,6):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all') #this needs to be changed when floats are forced to be integers 
    comp_out_path = os.path.join(out_pathway, 'volter2018intuitive_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'volter2018intuitive_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)